# California Housing Regression

This is a **standalone notebook** — run it directly without needing the `.py` files.
All model definitions and training logic are inlined below.

autoresearch uses the `.py` scripts (`model.py`, `train.py`); this notebook is for
interactive exploration and prototyping.

In [ ]:
import json
import os
import time

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

## Model Definition

Simple 2-hidden-layer MLP with ReLU activations for tabular regression.

In [ ]:
class TabularMLP(nn.Module):
    """
    Simple MLP for tabular regression.
    Input: (batch, input_dim)
    Output: (batch, output_dim)
    """

    def __init__(self, input_dim, output_dim=1, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, output_dim),
        )

    def forward(self, x):
        return self.net(x)


def make_model(input_dim, output_dim=1):
    return TabularMLP(input_dim=input_dim, output_dim=output_dim)

## Config & Device

In [ ]:
TIME_BUDGET = int(os.environ.get("AUTORESEARCH_TIME_BUDGET", 120))
BATCH_SIZE = 256
LR = 1e-3
WEIGHT_DECAY = 1e-5
NUM_WORKERS = 0
DATA_DIR = os.path.join("data", "california-housing")

if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.backends.cudnn.benchmark = True
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

## Load Data

In [ ]:
train_data = torch.load(os.path.join(DATA_DIR, "train.pt"), weights_only=True)
val_data = torch.load(os.path.join(DATA_DIR, "val.pt"), weights_only=True)

train_dataset = TensorDataset(train_data["x"], train_data["y"])
val_dataset = TensorDataset(val_data["x"], val_data["y"])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=NUM_WORKERS)

input_dim = train_data["x"].shape[1]
print(f"Train: {len(train_dataset)} samples, Val: {len(val_dataset)} samples")
print(f"Input features: {input_dim}")

## Train

In [ ]:
model = make_model(input_dim=input_dim, output_dim=1).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200)

num_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {num_params / 1e6:.4f}M")

t_start = time.time()
epoch = 0
best_val_mse = float("inf")

while True:
    elapsed = time.time() - t_start
    if elapsed >= TIME_BUDGET:
        break

    epoch += 1
    model.train()
    train_loss = 0.0
    train_count = 0

    for x_batch, y_batch in train_loader:
        if time.time() - t_start >= TIME_BUDGET:
            break
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        optimizer.zero_grad()
        pred = model(x_batch).squeeze(-1)
        loss = criterion(pred, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x_batch.size(0)
        train_count += x_batch.size(0)

    scheduler.step()

    model.eval()
    val_loss = 0.0
    val_count = 0
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            pred = model(x_batch).squeeze(-1)
            loss = criterion(pred, y_batch)
            val_loss += loss.item() * x_batch.size(0)
            val_count += x_batch.size(0)

    val_mse = val_loss / val_count if val_count > 0 else float("inf")
    train_mse = train_loss / train_count if train_count > 0 else float("inf")
    best_val_mse = min(best_val_mse, val_mse)
    print(f"Epoch {epoch}: train_mse={train_mse:.6f} val_mse={val_mse:.6f} lr={scheduler.get_last_lr()[0]:.6f}")

## Results

In [ ]:
total_time = time.time() - t_start

if device.type == "cuda":
    peak_memory_mb = torch.cuda.max_memory_allocated() / 1024 / 1024
elif device.type == "mps":
    peak_memory_mb = torch.mps.driver_allocated_size() / 1024 / 1024
else:
    peak_memory_mb = 0.0

metrics = {
    "val_mse": float(best_val_mse),
    "training_seconds": round(total_time, 1),
    "peak_memory_mb": round(peak_memory_mb, 1),
    "num_epochs": epoch,
    "num_params_M": round(num_params / 1e6, 4),
}

with open("metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Best val_mse: {best_val_mse:.6f}")
print(f"Training time: {total_time:.1f}s")
print(f"Peak memory: {peak_memory_mb:.1f} MB")
print(f"Metrics written to metrics.json")